In [1]:
# Install the YOLOv8 framework and Kaggle API
!pip install ultralytics kaggle


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.2 MB/s eta 0:00:00


In [2]:
# 1. Install the required YOLOv8 framework
!pip install ultralytics

# 2. Trigger the interactive file upload prompt
from google.colab import files
import os

print("Please select and upload your dataset zip file:")
uploaded = files.upload()

# Get the name of the file you uploaded dynamically
zip_name = list(uploaded.keys())[0]
print(f"\nSuccessfully uploaded: {zip_name}")

Please select and upload your dataset zip file:


Saving trafficsignsdataset.zip to trafficsignsdataset.zip

Successfully uploaded: trafficsignsdataset.zip


In [3]:
import zipfile

# Define the target extraction folder
extraction_folder = '/content/traffic_sign_dataset'

# Extract the uploaded zip file
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extraction_folder)

print(f"Dataset successfully unzipped into: {extraction_folder}")

Dataset successfully unzipped into: /content/traffic_sign_dataset


In [4]:
import yaml

# The true path to the data.yaml file inside the 'car' folder
yaml_path = '/content/traffic_sign_dataset/car/data.yaml'

# Load the file content
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

# Update the paths to match the exact nested directory structure in Colab
config['train'] = '/content/traffic_sign_dataset/car/train/images'
config['val'] = '/content/traffic_sign_dataset/car/valid/images'
config['test'] = '/content/traffic_sign_dataset/car/test/images'

# Save the corrected paths back to data.yaml
with open(yaml_path, 'w') as f:
    yaml.dump(config, f)

print("--- Updated data.yaml Configuration ---")
print(yaml.dump(config))

--- Updated data.yaml Configuration ---
names:
- Green Light
- Red Light
- Speed Limit 10
- Speed Limit 100
- Speed Limit 110
- Speed Limit 120
- Speed Limit 20
- Speed Limit 30
- Speed Limit 40
- Speed Limit 50
- Speed Limit 60
- Speed Limit 70
- Speed Limit 80
- Speed Limit 90
- Stop
nc: 15
roboflow:
  license: CC BY 4.0
  project: self-driving-cars-lfjou
  url: https://universe.roboflow.com/selfdriving-car-qtywx/self-driving-cars-lfjou/dataset/6
  version: 6
  workspace: selfdriving-car-qtywx
test: /content/traffic_sign_dataset/car/test/images
train: /content/traffic_sign_dataset/car/train/images
val: /content/traffic_sign_dataset/car/valid/images



In [5]:
from ultralytics import YOLO

# 1. Load the pre-trained Nano model weights for transfer learning
model = YOLO('yolov8n.pt')

# 2. Kick off the training loop using your exact data configuration
results = model.train(
    data='/content/traffic_sign_dataset/car/data.yaml', # Path to your fixed config file
    epochs=100,                                         # Assignment rule
    patience=20,                                        # Early stopping criteria
    batch=16,                                           # Ideal batch size for T4 GPU memory
    imgsz=640,                                          # Train resolution for small objects
    optimizer='SGD',                                    # Required optimizer
    lr0=0.01,                                           # Initial learning rate
    cos_lr=True,                                        # Cosine learning rate schedule
    device=0                                            # Force execution on the Colab GPU
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/traffic_sign_dataset/car/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, h

In [6]:
# Load your freshly trained custom model
best_model = YOLO('/content/runs/detect/train/weights/best.pt')

# Validate against the validation/test split
metrics = best_model.val()

# Print out your baseline metrics for your project report
print(f"Precision: {metrics.results_dict['metrics/precision(B)']:.4f}")
print(f"Recall: {metrics.results_dict['metrics/recall(B)']:.4f}")
print(f"mAP@50: {metrics.results_dict['metrics/mAP50(B)']:.4f}")

Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 392.9±106.3 MB/s, size: 16.0 KB)
val: Scanning /content/traffic_sign_dataset/car/valid/labels.cache... 801 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 801/801 186.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 5.9it/s 8.7s
                   all        801        944      0.942      0.949      0.972      0.838
           Green Light         87        122      0.875      0.836      0.884      0.544
             Red Light         74        108      0.841       0.83       0.84      0.495
       Speed Limit 100         52         52       0.98      0.953      0.993      0.893
       Speed Limit 110         17         17      0.761          1      0.995      0.909
       Speed Limit 120         6

In [7]:
# Load the model weights and export them to ONNX at 416x416 resolution

best_model = YOLO('/content/runs/detect/train/weights/best.pt')

# Exporting the model
onnx_path = best_model.export(format='onnx', imgsz=416)
print(f"Success! Optimized model saved to: {onnx_path}")

Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/detect/train/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 19, 3549) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 331ms
Prepared 4 packages in 1.94s
Installed 4 packages in 265ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.26.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 3.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with

In [8]:
# Run object tracking inference on your test video at the Pi-optimized resolution
video_results = best_model.predict(
    source='/content/traffic_sign_dataset/video.mp4',
    imgsz=416,
    save=True
)
print("Demo video processing complete!")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/508) /content/traffic_sign_dataset/video.mp4: 416x416 1 Stop, 5.9ms
video 1/1 (frame 2/508) /content/traffic_sign_dataset/video.mp4: 416x416 1 Stop, 6.7ms
video 1/1 (frame 3/508) /content/traffic_sign_dataset/video.mp4: 416x416 1 Stop, 5.9ms
video 1/1 (frame 4/508) /content/traffic_sign_dataset/video.mp4: 416x416 (no detections), 5.8ms
video 1/1 (frame 5/508) /content/traffic_sign_dataset/video.mp4: 416x416 1 Stop, 7.1ms
video 1/1 (fra

In [9]:
from google.colab import files
import os

# 1. Download the required model weights file
if os.path.exists('/content/runs/detect/train/weights/best.pt'):
    files.download('/content/runs/detect/train/weights/best.pt')
    print("Downloading best.pt...")

# 2. Download your optimized ONNX file (good for backup)
if os.path.exists('/content/runs/detect/train/weights/best.onnx'):
    files.download('/content/runs/detect/train/weights/best.onnx')
    print("Downloading best.onnx...")

# 3. Download your processed demonstration video file
# Note: YOLO normally names this video 'video.avi' or 'video.mp4' depending on openCV
if os.path.exists('/content/runs/detect/predict/video.mp4'):
    files.download('/content/runs/detect/predict/video.mp4')
elif os.path.exists('/content/runs/detect/predict/video.avi'):
    files.download('/content/runs/detect/predict/video.avi')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>